In [21]:
import httpx
import requests
from urllib.parse import urlencode, quote
import logging

In [22]:
logger = logging.getLogger(__name__)

In [23]:
response = requests.get("https://arxiv.org/abs/2305.16216")

### Get Metadats of papers from Arxiv

In [24]:
def get_arxiv_url() -> str:
    params = {
            "search_query": "cat:cs.AI",
            "start": 0,
            "max_results": 1,
            "sortBy": "submittedDate",
            "sortOrder": "descending",
        }
    url = f"https://export.arxiv.org/api/query?{urlencode(params, quote_via=quote, safe=":+[]")}"
    return url


In [25]:
async def get_papers_from_arxiv() -> str:
    try:
        url = get_arxiv_url()

        async with httpx.AsyncClient(timeout=60) as httpclient:
            response = await httpclient.get(url)
            response.raise_for_status()
            return response.text

    except httpx.TimeoutException:
        logger.exception("Timeout while calling arXiv API")
        raise

    except httpx.HTTPStatusError:
        logger.exception("arXiv API returned an error response")
        raise

    except Exception:
        logger.exception("Unexpected error occurred while calling arXiv API")
        raise

In [26]:
result = await get_papers_from_arxiv()
print(result)

<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/HQttLwSI9ar2vqfmj4lJ7u3JUas</id>
  <title>arXiv Query: search_query=cat:cs.AI&amp;id_list=&amp;start=0&amp;max_results=1</title>
  <updated>2026-06-17T09:23:12Z</updated>
  <link href="https://arxiv.org/api/query?search_query=cat:cs.AI&amp;start=0&amp;max_results=1&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>1</opensearch:itemsPerPage>
  <opensearch:totalResults>185055</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/2606.18247v1</id>
    <title>Visual Verification Enables Inference-time Steering and Autonomous Policy Improvement</title>
    <updated>2026-06-16T17:59:04Z</updated>
    <link href="https://arxiv.org/abs/2606.18247v1" rel="alternate" type="text/html"/>
    <link href="https

### Parse XML data to ArxivPaperAPI

In [7]:
from pydantic import BaseModel, HttpUrl
from typing import List
from datetime import datetime

class ArxivPaperAPI(BaseModel):
    title: str
    authors: List[str]
    arxiv_id: str
    summary: str
    categories: List[str]
    published_at: datetime
    pdf_url: HttpUrl

In [11]:
import logging
import xml.etree.ElementTree as ET
from datetime import datetime
from typing import List, Optional

logger = logging.getLogger(__name__)


class ArxivXmlParser:
    """
    Parse arXiv Atom XML responses into ArxivPaper objects.
    """

    def __init__(self):
        self._namespaces : dict = {
        "atom": "http://www.w3.org/2005/Atom",
        "opensearch": "http://a9.com/-/spec/opensearch/1.1/",
        "arxiv": "http://arxiv.org/schemas/atom",
    }

    def parse(self, xml_data: str) -> List[ArxivPaperAPI]:
        """
        Parse an arXiv XML response into a list of papers.
        """
        try:
            root = ET.fromstring(xml_data)
            entries = root.findall("atom:entry", self._namespaces)

            papers = []

            for entry in entries:
                paper = self._parse_entry(entry)

                if paper:
                    papers.append(paper)

            return papers

        except ET.ParseError as e:
            logger.error(f"Malformed XML from arXiv: {e}")
            raise

        except Exception as e:
            logger.exception("Unexpected error parsing arXiv response")
            raise
        
    def _parse_entry(self, entry: ET.Element) -> Optional[ArxivPaperAPI]:
        try:
            arxiv_id = self._get_id(entry)
            title = self._get_text(entry, "atom:title")
            summary = self._get_text(entry, "atom:summary")
            categories = self._get_categories(entry)
            published_at = self._get_published_at(entry)
            authors = self._get_authors(entry)
            pdf_url = self._get_pdf_url(entry)
            
            if not arxiv_id:
                logger.warning("Skipping entry with missing arXiv ID")
                return None

            return ArxivPaperAPI(
                title= title,
                authors=authors,
                arxiv_id=arxiv_id,
                summary=summary,
                categories=categories,
                published_at=published_at,
                pdf_url=pdf_url,
            )

        except Exception:
            logger.exception("Failed to parse arXiv entry")
            return None

    def _get_text(self, element: ET.Element, path: str) -> str:
        elem = element.find(path, self._namespaces)

        if elem is None or elem.text is None:
            return ""

        text = elem.text.strip()
        text = " ".join(text.split())

        return text

    def _get_id(self, entry: ET.Element) -> Optional[str]:
        id_elem = entry.find("atom:id", self._namespaces)

        if id_elem is None or id_elem.text is None:
            return None

        return id_elem.text.strip().split("/")[-1]

    def _get_authors(self, entry: ET.Element) -> List[str]:
        authors = []

        for author in entry.findall("atom:author", self._namespaces):
            name = self._get_text(author, "atom:name")

            if name:
                authors.append(name)

        return authors

    def _get_categories(self,entry: ET.Element) -> List[str]:
        categories = []

        for category in entry.findall("atom:category", self._namespaces):
            term = category.get("term")

            if term:
                categories.append(term)

        return categories

    def _get_published_at(self, entry: ET.Element) -> datetime:
        published = self._get_text(
            entry,
            "atom:published",
        )

        return datetime.fromisoformat(published.replace("Z", "+00:00"))

    def _get_pdf_url(self, entry: ET.Element) -> str:
        for link in entry.findall("atom:link", self._namespaces):
            if link.get("type") == "application/pdf":
                url = link.get("href", "")

                if url.startswith("http://arxiv.org/"):
                    url = url.replace(
                        "http://arxiv.org/",
                        "https://arxiv.org/",
                    )

                return url

        return ""

In [27]:
def parse_xml_to_arxivpaperapi(xml_data:str) -> List[ArxivPaperAPI]:
    xml_parser = ArxivXmlParser()
    response = xml_parser.parse(xml_data=xml_data)
    
    return response

result = await get_papers_from_arxiv()
list_arxiv_paper = parse_xml_to_arxivpaperapi(result)


In [28]:
for paper in list_arxiv_paper:
    print(f"{paper.arxiv_id}\n{paper.title}\n{paper.authors}\n{paper.categories}\n{paper.summary}\n{paper.published_at}\n{paper.pdf_url}")

2606.18247v1
Visual Verification Enables Inference-time Steering and Autonomous Policy Improvement
['Mingtong Zhang', 'Dhruv Shah']
['cs.RO', 'cs.AI']
Robots deployed in the real world should learn from their experience and improve over time. This requires a mechanism of practicing and learning from feedback. In this paper, we propose VERITAS, a generator-verifier framework for generalist robot policies for inference-time policy steering and self-improvement. We use a pre-trained generalist robot policy as a ``generator'' and pair it with a gradient-free ``visual verifier'' that evaluates actions at inference time. This framework enables inference-time steering that improves policy performance without additional training. We demonstrate that inference-time verification consistently outperforms vanilla generalists without training on additional demonstration data. Additionally, we demonstrate that the verified rollouts provide effective supervision for offline policy improvement: polici

After converting ArxivAPIPaper 
it should now be having pdf_url to get the paper downloaded. Again call get endpoint.
After downloading, docling helps to parse the paper. 
once it is parsed then store in database.

In [1]:
#call to arxiv with 3 sec apart 
from docling.document_converter import DocumentConverter

ARXIV_PAPER = "https://arxiv.org/pdf/1706.03762"
converter = DocumentConverter()

result = converter.convert(ARXIV_PAPER)


[INFO] 2026-06-17 15:02:37,747 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-17 15:02:37,753 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-17 15:02:37,768 [RapidOCR] download_file.py:60: File exists and is valid: /Users/meghaukkali/Documents/Academic-Paper-Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-17 15:02:37,769 [RapidOCR] main.py:50: Using /Users/meghaukkali/Documents/Academic-Paper-Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-17 15:02:37,984 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-17 15:02:37,984 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-06-17 15:02:37,987 [RapidOCR] download_file.py:60: File exists and is valid: /Users/meghaukkali/Documents/Academic-Paper-Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-06-17 15:02:37,987 [RapidOC

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [2]:
print(type(result))
print(type(result.document))

<class 'docling.datamodel.document.ConversionResult'>
<class 'docling_core.types.doc.document.DoclingDocument'>


In [3]:
for item in result.document.texts:
    print(item.label, item.text)

page_header arXiv:1706.03762v7  [cs.CL]  2 Aug 2023
text Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
section_header Attention Is All You Need
text Ashish Vaswani ∗ Google Brain avaswani@google.com Noam Shazeer ∗ Google Brain noam@google.com
text Llion Jones ∗ Google Research llion@google.com Niki Parmar ∗ Google Research nikip@google.com Aidan N. Gomez ∗ † University of Toronto aidan@cs.toronto.edu Jakob Uszkoreit ∗ Google Research usz@google.com Łukasz Kaiser ∗ Google Brain lukaszkaiser@google.com
text ∗ ‡
text Illia Polosukhin
text illia.polosukhin@gmail.com
section_header Abstract
text The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network a

In [ ]:
do asyncio to both downloading : save in local direcoty
do parsing with async : parse and return arxiv metadata and pdf content: set minimum page length and bytes in config.
store in postgre


### Download PDF from ARXIV

In [4]:
from pathlib import Path
def get_cache_path() -> Path:
    cache_dir = Path("./data/arxiv_pdfs")
    print(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir


In [49]:
import httpx

async def download_pdf(arxiv_paper: ArxivPaperAPI):
    safe_filename = arxiv_paper.arxiv_id.replace("/", "_") + ".pdf"
    cache_path =  get_cache_path() / safe_filename
    try:
        async with httpx.AsyncClient(timeout=60) as http_client:
            async with http_client.stream("GET", str(arxiv_paper.pdf_url)) as response:
                response.raise_for_status()
   
                try:
                    with open(cache_path, "wb") as file:
                        async for chunk in response.aiter_bytes():
                            file.write(chunk)
                except Exception as e:
                    logger.error("Issue occured when downloading", e)
                    raise
    except httpx.TimeoutException as te:
        logger.error("TimeoutError")
        raise
    except Exception as e:
        logger.error("Error")
        raise
    

In [50]:
import asyncio

for arxiv_paper in list_arxiv_paper:
    print(arxiv_paper.arxiv_id)
    print(arxiv_paper.pdf_url)
    await download_pdf(arxiv_paper)

2606.18247v1
https://arxiv.org/pdf/2606.18247v1
data/arxiv_pdfs


In [ ]:
from pathlib import Path
import asyncio

def pdf_cache_dir() -> Path:
       
    cache_dir = Path("./data/arxiv_pdfs")
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir

async def _apply_rate_limit(self) -> None:
        async with self._rate_limit_lock:
            if self._last_request_time is not None:
                elapsed = time.time() - self._last_request_time
                remaining = self.rate_limit_delay - elapsed

                if remaining > 0:
                    logger.debug(f"Rate limiting: waiting {remaining:.1f}s")
                    await asyncio.sleep(remaining)

            self._last_request_time = time.time()
    
async def download_pdf_using_pdf_url(arxiv_papers: ArxivPaperAPI):
    """Save pdfs in local"""
     
    for paper in arxiv_papers:
        safe_filename = paper.arxiv_id.replace("/", "_") + ".pdf"
        cache_path =  pdf_cache_dir() / safe_filename
        
        await _apply_rate_limit()

        for attempt in range(3):
            try:
                async with httpx.AsyncClient(timeout=60) as client:
                    async with client.stream("GET", paper.pdf_url) as response:
                        response.raise_for_status()

                        # FIX: wrap file write in try/finally so partial
                        # downloads are always cleaned up if something fails
                        try:
                            with open(cache_path, "wb") as f:
                                async for chunk in response.aiter_bytes():
                                    f.write(chunk)
                        except Exception:
                            # Delete partial file before re-raising
                            if cache_path.exists():
                                cache_path.unlink()
                                logger.warning(f"Deleted partial download: {cache_path.name}")
                            raise

                logger.info(f"Successfully downloaded to {cache_path.name}")
                return True

            except httpx.TimeoutException as e:
                # Linear backoff: wait 1× delay on attempt 1,
                # 2× on attempt 2, 3× on attempt 3, etc.
                # FIX: comment now matches actual math (was "exponential")
                wait_time = 5.0 * (attempt + 1)

                if attempt < 3 - 1:
                    logger.warning(
                        f"Timeout on attempt {attempt + 1}/{3}. "
                        f"Retrying in {wait_time}s..."
                    )
                    await asyncio.sleep(wait_time)
                else:
                    logger.error(f"Download timed out after {3} attempts")
                    raise

            except httpx.HTTPError as e:
                wait_time =  5.0 * (attempt + 1)

                if attempt < 3 - 1:
                    logger.warning(
                        f"HTTP error on attempt {attempt + 1}/{3}. "
                        f"Retrying in {wait_time}s..."
                    )
                    await asyncio.sleep(wait_time)
                else:
                    logger.error(f"Download failed after {3} attempts: {e}")
                    raise

            except Exception as e:
                # Unexpected error — don't retry, fail immediately
                logger.error(f"Unexpected download error: {e}")
                raise

            

### Parse downloaded Pdfs